# Christmas stock plan

This notebook builds Wilderness's Christmas stock plan from the data loaded at setup, then finds the Basecamp members to offer sleeping gear to. It creates three objects:

- `LAB2_ANALYTICS.TENT_RETURN_REASONS`: every Summit 2P return comment since 1 July, sorted into a reason by `AI_CLASSIFY`.
- `LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK`: one row per store and Christmas catalogue product, with the stock at risk and what to do about it.
- `LAB2_ANALYTICS.KIT_OFFER_AUDIENCE`: one row per Basecamp member who bought a tent from 1 to 3 November without sleeping gear, and whether to send them an offer.

The last cell writes one member's offer email with `AI_COMPLETE`. The Christmas planner app writes the rest on demand, with the same prompt.

Today is Wednesday 4 November 2026. The data runs to Tuesday 3 November. The Christmas period is 1 November to 24 December.

In [ ]:
-- Point the session at the workshop
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WILDERNESS_WH;
USE DATABASE WILDERNESS_HOL;
USE SCHEMA LAB2_ANALYTICS;

## 1. Stock at 3 November

Stock is held by store, product and batch. Most products have one batch. The Summit 2P has two: `B26-02` and `B26-07`.

In [ ]:
SELECT STORE_NAME, STATE, PRODUCT_NAME, BATCH_CODE, ON_HAND_UNITS, ON_ORDER_UNITS, AVG_WEEKLY_UNITS_LY_XMAS
FROM WILDERNESS_HOL.RAW.V_INVENTORY
WHERE IN_CHRISTMAS_CATALOGUE AND PRODUCT_ID = 'SKU-TNT-0102'
ORDER BY ON_HAND_UNITS DESC
LIMIT 10;

The forecast to Christmas Eve is last Christmas's weekly sales, times 51 days, times how this year's launch weekend compares with last year's for that state and category.

In [ ]:
SELECT s.STATE, p.CATEGORY,
       ROUND(SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0))
             / NULLIF(SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)), 0), 2) AS DEMAND_INDEX
FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
WHERE p.CATEGORY = 'Tents & Shelters'
  AND (f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03')
GROUP BY 1, 2
ORDER BY DEMAND_INDEX DESC;

## 2. Why the Summit 2P comes back

Stores record a coarse reason code on each return. The comment says more. `AI_CLASSIFY` reads every Summit 2P return comment since 1 July and gives it one of five reasons.

In [ ]:
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS AS
SELECT FEEDBACK_ID, PRODUCT_ID, BATCH_CODE, FEEDBACK_DATE,
       AI_CLASSIFY(FEEDBACK_TEXT, ['POLE_OR_FRAME_FAULT', 'SEAM_OR_WATERPROOFING_FAULT', 'COULD_NOT_PITCH',
                                   'CHANGE_OF_MIND', 'OTHER']):labels[0]::VARCHAR AS REASON
FROM WILDERNESS_HOL.RAW.CUSTOMER_FEEDBACK
WHERE PRODUCT_ID = 'SKU-TNT-0102'
  AND FEEDBACK_SOURCE = 'RETURN_COMMENT'
  AND FEEDBACK_DATE >= '2026-07-01';

In [ ]:
SELECT BATCH_CODE, REASON, COUNT(*) AS COMMENTS,
       ROUND(RATIO_TO_REPORT(COUNT(*)) OVER (PARTITION BY BATCH_CODE), 3) AS SHARE_OF_BATCH
FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

## 3. Which batch is faulty

The estimated defect rate is the batch's return rate times the share of its comments that describe a pole or seam fault. The Kestrel contract allows unsold stock of a batch to go back when that rate is above 8%.

In [ ]:
WITH rate AS (
  SELECT f.BATCH_CODE, SUM(COALESCE(r.RETURN_QUANTITY, 0)) / SUM(f.QUANTITY) AS RETURN_RATE
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  LEFT JOIN WILDERNESS_HOL.RAW.FCT_RETURN r ON r.SALE_LINE_ID = f.SALE_LINE_ID
  WHERE f.PRODUCT_ID = 'SKU-TNT-0102' AND f.BATCH_CODE IN ('B26-02', 'B26-07')
  GROUP BY 1),
fault AS (
  SELECT BATCH_CODE, AVG(IFF(REASON IN ('POLE_OR_FRAME_FAULT', 'SEAM_OR_WATERPROOFING_FAULT'), 1, 0)) AS FAULT_SHARE
  FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
  GROUP BY 1)
SELECT rate.BATCH_CODE, ROUND(RETURN_RATE, 4) AS RETURN_RATE, ROUND(FAULT_SHARE, 3) AS FAULT_SHARE,
       ROUND(RETURN_RATE * FAULT_SHARE, 4) AS EST_DEFECT_RATE,
       RETURN_RATE * FAULT_SHARE > 0.08 AS CAN_GO_BACK_TO_KESTREL
FROM rate JOIN fault ON fault.BATCH_CODE = rate.BATCH_CODE
ORDER BY 1;

## 4. The stock plan

The view below holds the plan. A row is `STOCKOUT` when stock plus orders covers less than 6.5 weeks of the forecast, so the store runs out before about 16 December. It is `OVERSTOCK` when stock covers more than 7.5 weeks, so stock will be left on Boxing Day. Anything in between is `OK`. Excess stock goes back to the supplier first if it is from a faulty batch, then to a store that is short, and is marked down on Boxing Day only as a last resort.

In [ ]:
CREATE OR REPLACE VIEW WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
  COMMENT = 'Christmas stock plan at 3 Nov 2026: one row per store and Christmas catalogue product'
AS
WITH idx AS (
  SELECT s.STATE, p.CATEGORY,
         SUM(IFF(f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03', f.QUANTITY, 0))
         / NULLIF(SUM(IFF(f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03', f.QUANTITY, 0)), 0) AS IDX
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = f.STORE_ID
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-11-03' OR f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03'
  GROUP BY 1, 2),
batch_rate AS (
  SELECT f.PRODUCT_ID, f.BATCH_CODE, SUM(COALESCE(r.RETURN_QUANTITY, 0)) / SUM(f.QUANTITY) AS RETURN_RATE
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
  LEFT JOIN WILDERNESS_HOL.RAW.FCT_RETURN r ON r.SALE_LINE_ID = f.SALE_LINE_ID
  WHERE p.CATEGORY = 'Tents & Shelters' AND f.BATCH_CODE IS NOT NULL
  GROUP BY 1, 2),
fault_share AS (
  SELECT PRODUCT_ID, BATCH_CODE,
         AVG(IFF(REASON IN ('POLE_OR_FRAME_FAULT', 'SEAM_OR_WATERPROOFING_FAULT'), 1, 0)) AS FAULT_SHARE
  FROM WILDERNESS_HOL.LAB2_ANALYTICS.TENT_RETURN_REASONS
  GROUP BY 1, 2),
batch_defect AS (
  SELECT b.PRODUCT_ID, b.BATCH_CODE, b.RETURN_RATE * fs.FAULT_SHARE AS EST_DEFECT_RATE
  FROM batch_rate b
  JOIN fault_share fs ON fs.PRODUCT_ID = b.PRODUCT_ID AND fs.BATCH_CODE = b.BATCH_CODE),
inv AS (
  SELECT i.STORE_ID, i.PRODUCT_ID,
         SUM(i.ON_HAND_UNITS) AS ON_HAND_UNITS,
         SUM(i.ON_ORDER_UNITS) AS ON_ORDER_UNITS,
         SUM(i.AVG_WEEKLY_UNITS_LY_XMAS) AS AVG_WEEKLY,
         SUM(IFF(d.EST_DEFECT_RATE > 0.08, i.ON_HAND_UNITS, 0)) AS RETURNABLE_UNITS,
         MAX(IFF(i.ON_HAND_UNITS > 0, d.EST_DEFECT_RATE, NULL)) AS EST_DEFECT_RETURN_RATE
  FROM WILDERNESS_HOL.RAW.FCT_INVENTORY i
  LEFT JOIN batch_defect d ON d.PRODUCT_ID = i.PRODUCT_ID AND d.BATCH_CODE = i.BATCH_CODE
  GROUP BY 1, 2),
base AS (
  SELECT s.STORE_ID, s.STORE_NAME, s.STATE, s.LATITUDE, s.LONGITUDE,
         p.PRODUCT_ID, p.PRODUCT_NAME, p.KIT_GROUP, p.UNIT_COST,
         v.ON_HAND_UNITS, v.ON_ORDER_UNITS, v.RETURNABLE_UNITS, v.EST_DEFECT_RETURN_RATE,
         v.ON_HAND_UNITS + v.ON_ORDER_UNITS AS STOCK,
         v.AVG_WEEKLY * COALESCE(x.IDX, 1) AS WEEKLY_FC
  FROM inv v
  JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = v.STORE_ID
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = v.PRODUCT_ID
  LEFT JOIN idx x ON x.STATE = s.STATE AND x.CATEGORY = p.CATEGORY
  WHERE p.IN_CHRISTMAS_CATALOGUE),
classed AS (
  SELECT base.*,
         WEEKLY_FC * 51 / 7 AS FORECAST,
         STOCK / NULLIF(WEEKLY_FC, 0) AS COVER,
         CASE WHEN STOCK / NULLIF(WEEKLY_FC, 0) < 6.5 THEN 'STOCKOUT'
              WHEN STOCK / NULLIF(WEEKLY_FC, 0) > 7.5 THEN 'OVERSTOCK'
              ELSE 'OK' END AS RISK
  FROM base)
SELECT
  STORE_ID, STORE_NAME, STATE, LATITUDE, LONGITUDE, PRODUCT_ID, PRODUCT_NAME, KIT_GROUP, UNIT_COST,
  ON_HAND_UNITS, ON_ORDER_UNITS, RETURNABLE_UNITS,
  ROUND(FORECAST, 1) AS FORECAST_UNITS_TO_XMAS,
  ROUND(COVER, 1) AS WEEKS_OF_COVER,
  ROUND(EST_DEFECT_RETURN_RATE, 4) AS EST_DEFECT_RETURN_RATE,
  RISK,
  ROUND(IFF(RISK = 'OVERSTOCK', GREATEST(0, STOCK - FORECAST), 0), 0) AS EXCESS_UNITS,
  ROUND(IFF(RISK = 'STOCKOUT', GREATEST(0, FORECAST - STOCK), 0), 0) AS SHORTFALL_UNITS,
  ROUND(IFF(RISK = 'OVERSTOCK', GREATEST(0, STOCK - FORECAST), 0) * UNIT_COST, 2) AS STOCK_AT_RISK_AUD,
  CASE
    WHEN RISK = 'OVERSTOCK' AND RETURNABLE_UNITS > 0 THEN 'RETURN TO SUPPLIER'
    WHEN RISK = 'OVERSTOCK' AND COUNT_IF(RISK = 'STOCKOUT') OVER (PARTITION BY PRODUCT_ID) > 0 THEN 'TRANSFER OUT'
    WHEN RISK = 'OVERSTOCK' THEN 'MARK DOWN BOXING DAY'
    WHEN RISK = 'STOCKOUT' AND COUNT_IF(RISK = 'OVERSTOCK') OVER (PARTITION BY PRODUCT_ID) > 0 THEN 'TRANSFER IN'
    WHEN RISK = 'STOCKOUT' THEN 'REORDER BY 13 NOV'
    ELSE 'NONE'
  END AS SUGGESTED_ACTION
FROM classed;

The ten rows with the most stock at risk:

In [ ]:
SELECT STORE_NAME, STATE, PRODUCT_NAME, RISK, EXCESS_UNITS, RETURNABLE_UNITS, STOCK_AT_RISK_AUD, SUGGESTED_ACTION
FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
ORDER BY STOCK_AT_RISK_AUD DESC, STORE_ID, PRODUCT_ID
LIMIT 10;

Stock at risk by state. Switch the result to **Chart** to see it as a bar chart.

In [ ]:
SELECT STATE, ROUND(SUM(STOCK_AT_RISK_AUD)) AS STOCK_AT_RISK_AUD,
       COUNT_IF(RISK = 'STOCKOUT') AS STOCKOUT_ROWS
FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
GROUP BY 1
ORDER BY 2 DESC;

## 5. The plan in one paragraph

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
row = session.sql("""
    SELECT
      (SELECT SUM(STOCK_AT_RISK_AUD) FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
        WHERE KIT_GROUP = 'SHELTER') AS TENT_AUD,
      (SELECT SUM(STOCK_AT_RISK_AUD) FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
        WHERE KIT_GROUP = 'SHELTER' AND STATE IN ('VIC', 'TAS')) AS TENT_AUD_VIC_TAS,
      (SELECT SUM(LEAST(RETURNABLE_UNITS, EXCESS_UNITS) * UNIT_COST) FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
        WHERE SUGGESTED_ACTION = 'RETURN TO SUPPLIER') AS RETURNABLE_AUD,
      (SELECT COUNT(*) FROM WILDERNESS_HOL.LAB2_ANALYTICS.V_CHRISTMAS_STOCK_RISK
        WHERE PRODUCT_ID = 'SKU-TNT-0103' AND RISK = 'STOCKOUT') AS S3P_STORES,
      (SELECT COUNT(*) FROM WILDERNESS_HOL.RAW.V_KIT_COVER
        WHERE KIT_GROUP = 'SLEEP' AND RUN_OUT_WEEK = '2026-12-07') AS SLEEP_STORES
""").collect()[0]

print(
    f"AUD {row['TENT_AUD']:,.0f} of tent stock will still be on shelves on Boxing Day, "
    f"AUD {row['TENT_AUD_VIC_TAS']:,.0f} of it in Victoria and Tasmania. "
    f"AUD {row['RETURNABLE_AUD']:,.0f} is the faulty Summit 2P batch and can go back to Kestrel. "
    f"Meanwhile {row['S3P_STORES']} stores run out of the Summit 3P, and {row['SLEEP_STORES']} stores run out "
    f"of sleeping gear in the week of 7 December, unless we reorder by 13 November."
)

## 6. The sleeping-gear offer for Basecamp members

Six in ten tent orders last Christmas also had sleeping gear. Some of this week's tent buyers left without it, and the Basecamp members among them can be sent an offer. The offer only goes where the store has sleeping gear to sell: in a store that is already short, it would make the shortage worse. Online orders use the member's home store.

In [ ]:
CREATE OR REPLACE TABLE WILDERNESS_HOL.LAB2_ANALYTICS.KIT_OFFER_AUDIENCE AS
WITH tent_orders AS (
  SELECT f.ORDER_ID, f.ORDER_DATE, f.MEMBER_ID, f.STORE_ID,
         MAX(IFF(p.CATEGORY = 'Tents & Shelters', p.PRODUCT_NAME, NULL)) AS TENT_NAME,
         MAX(IFF(p.KIT_GROUP = 'SLEEP', 1, 0)) AS HAS_SLEEP
  FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
  JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
  WHERE f.ORDER_DATE BETWEEN '2026-11-01' AND '2026-11-03' AND f.MEMBER_ID IS NOT NULL
  GROUP BY 1, 2, 3, 4),
short AS (
  SELECT STORE_ID FROM WILDERNESS_HOL.RAW.V_KIT_COVER
  WHERE KIT_GROUP = 'SLEEP' AND KIT_SHORTFALL_UNITS > 0)
SELECT m.MEMBER_ID, m.FIRST_NAME, m.TIER, t.TENT_NAME,
       s.STORE_ID, s.STORE_NAME, s.STATE,
       sh.STORE_ID IS NOT NULL AS STORE_SHORT_OF_SLEEP,
       sh.STORE_ID IS NULL AS SEND_OFFER
FROM tent_orders t
JOIN WILDERNESS_HOL.RAW.DIM_MEMBER m ON m.MEMBER_ID = t.MEMBER_ID
JOIN WILDERNESS_HOL.RAW.DIM_STORE s ON s.STORE_ID = COALESCE(NULLIF(t.STORE_ID, 'ONLINE'), m.HOME_STORE_ID)
LEFT JOIN short sh ON sh.STORE_ID = s.STORE_ID
WHERE t.TENT_NAME IS NOT NULL AND t.HAS_SLEEP = 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY m.MEMBER_ID ORDER BY t.ORDER_DATE, t.ORDER_ID) = 1;

The opportunity, if each member who gets an offer spends what a tent order with sleeping gear spent on it last Christmas:

In [ ]:
WITH ly AS (
  SELECT AVG(SLEEP_SPEND) AS AVG_SLEEP_SPEND FROM (
    SELECT f.ORDER_ID,
           SUM(IFF(p.KIT_GROUP = 'SLEEP', f.NET_SALES_AMOUNT, 0)) AS SLEEP_SPEND,
           MAX(IFF(p.CATEGORY = 'Tents & Shelters', 1, 0)) AS HAS_TENT
    FROM WILDERNESS_HOL.RAW.FCT_SALES_LINE f
    JOIN WILDERNESS_HOL.RAW.DIM_PRODUCT p ON p.PRODUCT_ID = f.PRODUCT_ID
    WHERE f.ORDER_DATE BETWEEN '2025-11-01' AND '2025-12-24'
    GROUP BY 1)
  WHERE HAS_TENT = 1 AND SLEEP_SPEND > 0)
SELECT COUNT(*) AS MEMBERS,
       COUNT_IF(SEND_OFFER) AS OFFERS,
       COUNT_IF(NOT SEND_OFFER) AS HELD_BACK,
       ROUND(MAX(ly.AVG_SLEEP_SPEND), 2) AS AVG_SLEEP_SPEND_LY,
       ROUND(COUNT_IF(SEND_OFFER) * MAX(ly.AVG_SLEEP_SPEND)) AS OPPORTUNITY_AUD
FROM WILDERNESS_HOL.LAB2_ANALYTICS.KIT_OFFER_AUDIENCE
CROSS JOIN ly;

`AI_COMPLETE` writes the email. Check the wording on one member here; the app writes the email for any member at the press of a button, with the same prompt.

In [ ]:
SELECT FIRST_NAME, TENT_NAME, STORE_NAME,
       AI_COMPLETE('claude-sonnet-4-6',
         'Write a short email from Wilderness Outfitters, an Australian outdoor retailer, to a Basecamp loyalty member. '
         || 'They are ' || FIRST_NAME || ', who bought a ' || TENT_NAME || ' at ' || STORE_NAME
         || ' this week but no sleeping bag or mat. Thank them, and offer 15% off a sleeping bag or mat in store at '
         || STORE_NAME || ' before 24 December. Plain Australian English, under 80 words, no exclamation marks, '
         || 'no emoji. Return only a subject line and the body.')::VARCHAR AS EMAIL
FROM WILDERNESS_HOL.LAB2_ANALYTICS.KIT_OFFER_AUDIENCE
WHERE SEND_OFFER
ORDER BY MEMBER_ID
LIMIT 1;